# 동의어 처리를 통한 First-Stage Retrieve 성능 개선
* Custom 데이터셋 생성 시 표준 용어집을 활용하여 entity를 정규화함
    * Knowledge Graph 구축 과정에서 온톨로지 중심 entity(동작명, 신체 부위) 표준화
    * 정규화된 용어를 기반으로 RAGAS 합성 데이터셋 생성
* 실제 사용자 질의 및 문서와 동일한 표현에 대해 여러 표현을 사용
    * 예: "인상" ↔ "스내치", "Clean" ↔ "클린"
* 용어 불일치로 인한 검색 실패를 개선하기 위해 동의어 전처리 기능 도입
* 기존 최적 성능(k: 10, alpha: 40, bm25_kiwi_pos, threshold: 0.1 --> custom_recall@10: 0.571429)을 기준으로 실험

### 동의어 매칭(동의어 사전) 로직
1. 표준 용어집 기반 딕셔너리 구축
    * name_dict, body_dict를 통합한 dictionary
2. 정규화 기반 딕셔너리 생성
    * 소문자 전처리
    * kiwi 기반 주요 품사('NNG': 일반명사, 'NNP': 고유명사, 'NNB': 의존명사, 'SL': 외국어) 추출
3. 질의(user_input) kiwi 기반 주요 품사 추출 
4. 복합어(clean pull과 같이 두 개 이상의 명사로 구성된 것) -> 단일어 순으로 처리
5. 질의 동의어 추가
    * 원본 질의 예시: "스내치(Snatch) 기술에서 동적 시작 자세와 정적 시작 자세의 주요 차이점은 무엇인가요?"
    * 동의어 추가 예시: "스내치(Snatch) 기술에서 동적 시작 자세와 정적 시작 자세의 주요 차이점은 무엇인가요? 잡아채기 Snatch 인상 Snatch Pull 인상기술"

### 결과
* 단순 동의어 처리를 통해 custom dataset에 대한 recall@10 성능 지표 향상을 기대했으나, 오히려 저하됨
* 과도한 동의어 추가로 쿼리 의도 희석
* 동의어 전처리를 통한 recall@10 개선 효과: 0.571429 → 0.482142%
* 동의어 매칭의 일반적인 적용과 다른 단어 사용 방식으로 인한 성능 하락
    * 일반적으로 동의어 매칭은 '어르신 <-> 노인'과 같은 단어를 위해서 사용됨
    * 본 프로젝트에선 'Snatch <-> 스내치'와 같이 영어로 변경되는 단어들을 위해 사용됨
    

In [68]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

from typing import List, Dict
from pydantic import BaseModel, Field
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

from kiwipiepy import Kiwi

In [67]:
import sys
sys.path.append('../code/graphParser')
sys.path.append('../code/ragas_custom')
from rateLimit import handle_rate_limits
from retrieve.sparse import BM25
from retrieve.config import generate_retriever_configs
from evaluation.retrieve import optimization, combine_hybrid_results, evaluate_metrics

from langchain_core.prompts import load_prompt
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaLLM

from ragas.testset.graph import KnowledgeGraph

from dotenv import load_dotenv
load_dotenv()

True

In [69]:
from langchain_core.documents import Document

kg = KnowledgeGraph.load('../data/rag/kg.json')

documents = [Document(page_content=node.properties['page_content'],
                      metadata=node.properties['document_metadata'])
                       for node in kg.nodes]

In [70]:
standard_config = {
    'k': 10,
    'alpha': 40,
    'dense_type': 'threshold',
    'dense_params': {'score_threshold': 0.1},
    'morphological_analyzer': 'bm25_kiwi_pos'
}

In [71]:
kiwi_pos = BM25(k=15, type='kiwi_pos')
texts = [node.properties['page_content'] for node in kg.nodes]
kiwi_pos.from_texts(texts)

embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(documents, embeddings)

In [72]:
merged_dataset = pd.read_csv('../data/rag/valid_dataset.csv')
merged_dataset['reference_contexts'] = merged_dataset['reference_contexts'].apply(lambda x : eval(x))

for column in merged_dataset.columns[5:]:
    merged_dataset[column] = merged_dataset[column].apply(lambda x : eval(x))

In [83]:
name_dict = {
    "Snatch": "스내치",
    "snatch": "스내치",
    "Snatch high Pull": "스내치",
    "Snatch Pull": "스내치",
    "스내치 풀": "스내치",
    "잡아채기": "스내치",
    "인상": "스내치",
    "인상동작": "스내치",
    "인상기술": "스내치",
    
    "Clean": "클린",
    "clean": "클린",
    "Top Clean": "클린",
    "High Clean": "클린",
    "High Clean from Hang": "클린",
    "Clean Pull": "클린",
    "Clean high Pull": "클린",
    "클린 풀": "클린",
    "클린동작": "클린",
    
    "Jerk": "저크",
    "jerk": "저크",
    "Jerk Dip": "저크",
    "Jerk Balance": "저크",
    "Jerk Recovery": "저크",
    "Jerk Up": "저크",
    "Jerk Split": "저크",
    "저크 딥": "저크",
    "저크딥": "저크",
    "저크 업": "저크",
    "저크업": "저크",
    "저크 스프리트": "저크",
    "저크스플릿": "저크",
    "저크동작": "저크",

    "Clean and Jerk": "클린 앤 저크",
    "Clean & Jerk": "클린 앤 저크",
    "clean and jerk": "클린 앤 저크",
    "클린 앤 저크": "클린 앤 저크",
    "용상": "클린 앤 저크",

    "Front Squat": "프론트 스쿼트",
    "프론트 스쿼트": "프론트 스쿼트",

    "Back Squat": "백 스쿼트",
    "백 스쿼트": "백 스쿼트",

    "Squat": "스쿼트",
    "스쿼트": "스쿼트",
    "하프 스쿼트": "스쿼트",
    "하프-스쿼트": "스쿼트",

    "Power Clean": "파워 클린",
    "파워 클린": "파워 클린",

    "Power Snatch": "파워 스내치",
    "Power snatch": "파워 스내치",
    "파워 스내치": "파워 스내치",

    "Push Press": "푸시 프레스",
    "푸시 프레스": "푸시 프레스",

    "Push Jerk": "푸시 저크",
    "푸시 저크": "푸시 저크",

    "Split Jerk": "저크 스플릿",
    "저크 스플릿": "저크 스플릿",

    "Snatch Balance": "스내치 밸런스",
    "스내치 밸런스": "스내치 밸런스",

    "Clean Deadlift": "클린 데드리프트",
    "Straight Clean Deadlift": "클린 데드리프트",
    "데드 리프트": "클린 데드리프트",

    "Snatch Deadlift": "스내치 데드리프트",
    "Snatch Dead Lift": "스내치 데드리프트",
    "스내치 데드리프트": "스내치 데드리프트",

    "Good Morning-Exercise": "굿모닝",
    "Good Morning": "굿모닝",

    "Military Press": "밀리터리 프레스",
    "밀리터리 프레스": "밀리터리 프레스",

    "One Arm Dumbbell Row": "원암 덤벨 로우",
    "One Hand Clean": "원암 덤벨 로우",
    "Bent-Over Row": "원암 덤벨 로우",

    "Leg Curl": "레그 컬",
    "레그 컬": "레그 컬",

    "Leg Press": "레그 프레스",
    "레그 프레스": "레그 프레스",

    "Sit-up": "싯업",
    "윗몸일으키기": "싯업",
    "싯업": "싯업",

    "Hyper-extension": "하이퍼 익스텐션",
    "Back Hyper-extension": "하이퍼 익스텐션",
    "하이퍼 익스텐션": "하이퍼 익스텐션",


    "Bench Press": "벤치 프레스",
    "Flat Barbell Bench Press": "벤치 프레스",
    "Incline Barbell Bench Press": "벤치 프레스",
    "벤치 프레스": "벤치 프레스",

    "바벨 컬": "바벨 컬",
    "컬": "바벨 컬",
    "트라이셉스 컬 바": "바벨 컬",

    "Pull-up": "턱걸이",
    "턱걸이": "턱걸이",

    "덤벨 레터럴 레이즈": "레터럴 레이즈",

    "Jump Snatch": "점프 스내치",
    "점프 스내치": "점프 스내치",

    "Jump Clean (on disk)": "점프 클린",
    "점프 클린": "점프 클린",
}

In [84]:
body_dict = {
    # 머리
    "얼굴": "머리", 

    # 목
    "목": "목", "경추": "목",

    # 어깨
    "삼각근": "어깨", "승모근": "어깨",

    # 팔
    "상완": "팔", "이두근": "팔", "삼두근": "팔",

    # 손목
    "손목": "팔",

    # 가슴
    "대흉근": "가슴", "소흉근": "가슴",

    # 몸통
    "몸": "몸통", "동체": "몸통",

    # 복부
    "복근": "복부", "복직근": "복부", "외복사근": "복부",

    # 허리
    "허리근": "허리", "척추기립근": "허리",

    # 엉덩이
    "둔부": "엉덩이", "고관절": "엉덩이", "대둔근": "엉덩이",

    # 다리
    "대퇴": "다리", "허벅지": "다리", "햄스트링": "다리", "종아리": "다리",

    # 발
    "발목": "발", "발가락": "발",

    # 척추
    "척추": "척추",

    # 근육
    "골격근": "근육", "근섬유": "근육", "근력": "근육",
}

In [85]:
kiwi = Kiwi()

In [86]:
def normalize_with_kiwi(text: str, kiwi: Kiwi) -> str:
    """
    Kiwi를 사용하여 텍스트를 정규화
    """
    result = kiwi.analyze(text)
    nouns = []
    
    for token in result[0][0]:
        if token.tag in ['NNG', 'NNP', 'NNB', 'SL']:
            nouns.append(token.form.lower())
    
    return " ".join(nouns) if nouns else text.lower()

In [87]:
combined_dict = {**name_dict, **body_dict}
new_combined_dict = {}
normalized_to_original = {}  # 핵심: 정규화 → 원본 매핑

# Step 1: key와 value 모두 정규화하면서 원본 유지
for original_key, original_value in combined_dict.items():
    normalized_key = normalize_with_kiwi(original_key, kiwi)
    normalized_value = normalize_with_kiwi(original_value, kiwi)
    
    if normalized_key:
        # 매칭용: 정규화된 형태 저장
        if normalized_key not in new_combined_dict:
            new_combined_dict[normalized_key] = normalized_value
        
        # 역매핑: 정규화 → 원본 (나중에 원본을 복원하기 위해)
        if normalized_key not in normalized_to_original:
            normalized_to_original[normalized_key] = []
        normalized_to_original[normalized_key].append(original_key)

# Step 2: 표준용어 자기 자신도 추가
original_standards = {}  # {정규화된 표준용어: 원본 표준용어}

for standard_term in set(new_combined_dict.values()):
    # 원본 표준용어 찾기
    original_standard = None
    for orig_key, orig_val in combined_dict.items():
        if normalize_with_kiwi(orig_val, kiwi) == standard_term:
            original_standard = orig_val
            break
    
    if original_standard:
        original_standards[standard_term] = original_standard
    
    # 정규화된 표준용어를 키로도 추가
    new_combined_dict[standard_term] = standard_term
    if standard_term not in normalized_to_original:
        normalized_to_original[standard_term] = []
    if original_standard and original_standard not in normalized_to_original[standard_term]:
        normalized_to_original[standard_term].append(original_standard)

# Step 3: reverse_dict 생성 - 원본 형태 저장!
reverse_dict = {}

for normalized_standard in set(new_combined_dict.values()):
    reverse_dict[normalized_standard] = []
    
    # 이 표준용어로 매핑되는 모든 정규화 키 찾기
    for norm_key, norm_val in new_combined_dict.items():
        if norm_val == normalized_standard:
            # 원본 형태들을 추가
            if norm_key in normalized_to_original:
                reverse_dict[normalized_standard].extend(normalized_to_original[norm_key])
    
    # 중복 제거
    reverse_dict[normalized_standard] = list(set(reverse_dict[normalized_standard]))

In [ ]:
def synonym_process(user_input: str, new_combined_dict: Dict[str, str], 
                   reverse_dict: Dict[str, List[str]], kiwi: Kiwi) -> str:
    """
    정규화로 매칭하되, 원본 형태를 쿼리에 추가
    """
    matched_terms = set()
    
    # 1. Kiwi로 명사 추출
    result = kiwi.analyze(user_input)
    nouns = []
    for token in result[0][0]:
        if token.tag in ['NNG', 'NNP', 'NNB', 'SL']:
            nouns.append(token.form.lower())
    
    if not nouns:
        return user_input
    
    used_indices = set()
    
    # 2. 복합어→단일어 순서로 매칭 (정규화 기반)
    for length in range(3, 0, -1):
        for i in range(len(nouns) - length + 1):
            if any(idx in used_indices for idx in range(i, i + length)):
                continue
            
            compound = " ".join(nouns[i:i+length])
            
            # 정규화된 딕셔너리에서 조회
            if compound in new_combined_dict:
                standard_term = new_combined_dict[compound]
                matched_terms.add(standard_term)
                used_indices.update(range(i, i + length))
    
    # 3. 원본 동의어 수집
    all_synonyms = []
    for standard_term in matched_terms:
        if standard_term in reverse_dict:
            # reverse_dict에는 원본 형태가 저장되어 있음
            original_synonyms = reverse_dict[standard_term][:5]
            all_synonyms.extend(original_synonyms)
    
    # 4. 원본 쿼리 + 원본 동의어 반환
    if all_synonyms:
        return user_input + " " + " ".join(all_synonyms)
    return user_input

In [ ]:
merged_dataset = pd.read_csv('../data/rag/valid_dataset.csv')
merged_dataset['reference_contexts'] = merged_dataset['reference_contexts'].apply(lambda x : eval(x))

for column in merged_dataset.columns[5:]:
    merged_dataset[column] = merged_dataset[column].apply(lambda x : eval(x))

merged_dataset['synonym_input'] = merged_dataset.apply(lambda x : synonym_process(x['user_input'], new_combined_dict, reverse_dict, kiwi), axis=1)

merged_dataset['precompute_dense'] = merged_dataset['synonym_input'].apply(lambda x : db.similarity_search_with_score(x, k=int(15)))
merged_dataset['precompute_sparse_bm25_kiwi_pos'] = merged_dataset['synonym_input'].apply(lambda x : kiwi_pos.search(x))

merged_dataset['precompute_dense'] = merged_dataset['precompute_dense'].apply(lambda results: [doc.page_content for result in results for doc, score in [result] if score >= standard_config['dense_params']['score_threshold']])
merged_dataset['result_retrieve'] = combine_hybrid_results(merged_dataset['precompute_dense'], merged_dataset['precompute_sparse_bm25_kiwi_pos'], standard_config['alpha'], standard_config['k'])
merged_dataset['need_retrieve'] = merged_dataset.apply(lambda x : 'no' if len(set(x['reference_contexts']).intersection(set(x['result_retrieve']))) == len(x['reference_contexts']) else 'yes', axis=1)

merged_dataset = merged_dataset[merged_dataset.columns[:6].to_list() + ['result_retrieve', 'need_retrieve']]
# merged_dataset.to_csv('../data/rag/synonum.csv', index=False)

In [120]:
merged_dataset.head(2)

,user_input,reference_contexts,reference,synthesizer_name,reference_contexts_section,synonym_input,result_retrieve,need_retrieve
0,바벨 잡는 방법과 앉아받기 동작의 올바른 수행을 결합하여 최적의 리프팅 기술을 설명해줘.,"[바벨을 잡는 방법에는 크게 오버그립(over grip), 언더그립(under gr...","바벨 잡는 방법에는 오버그립, 언더그립, 리버스그립, 훅그립이 있으며, 각각의 그립...",multi_hop_abstract_query_synthesizer,"['Ⅲ. 역도경기 기술의 구조와 훈련법', 'Ⅲ. 역도경기 기술의 구조와 훈련법']",바벨 잡는 방법과 앉아받기 동작의 올바른 수행을 결합하여 최적의 리프팅 기술을 설명해줘.,[순간적으로 무거운 물체를 들어 올리는 데에는 근력 외에도 강인한 정신력이 요구\n...,yes
1,"스포츠 심리학자들은 선수의 성과 향상에 어떤 역할을 하며, 그들의 개인 문제에 대한...",[성공적인 상\n담 진행을 위해서 상담사는 내담자의 감정에 공감할 수 있어야 한다....,스포츠 심리학자들은 운동선수의 성과를 향상시키기 위해 여러 가지 역할을 수행합니다....,multi_hop_abstract_query_synthesizer,"['II. 역도의 스포츠 과학적 원리', 'II. 역도의 스포츠 과학적 원리']","스포츠 심리학자들은 선수의 성과 향상에 어떤 역할을 하며, 그들의 개인 문제에 대한...",[스포츠심리학을 연구하고 스포츠심리학의 연구 성과를 스포츠 현장에 적용하는\n스포츠...,no


In [122]:
evaluate_metrics(merged_dataset['result_retrieve'], merged_dataset['reference_contexts'], standard_config['k'])

{'ndcg': 0.49684447178860275,
 'recall': 0.6011904761904762,
 'map': 0.5578514739229025}

In [125]:
custom_dataset = merged_dataset.loc[merged_dataset['synthesizer_name'].isin(merged_dataset['synthesizer_name'].unique()[-3:])]

In [127]:
evaluate_metrics(custom_dataset['result_retrieve'], custom_dataset['reference_contexts'], standard_config['k'])

{'ndcg': 0.3926581099029988,
 'recall': 0.48214285714285715,
 'map': 0.5087159863945578}